# Question 17 - 13. Roman to Integer

Roman numerals are represented by seven different symbols: `I`, `V`, `X`, `L`, `C`, `D` and `M`.

| Symbol | Value |
|---|---|
| I | 1 |
| V | 5 |
| X | 10 |
| L | 50 |
| C | 100 |
| D | 500 |
| M | 1000 |

For example, `2` is written as `II` in Roman numeral, just two ones added together. `12` is written as `XII`, which is simply `X + II`. The number `27` is written as `XXVII`, which is `XX + V + II`.

Roman numerals are usually written largest to smallest from left to right. However, the numeral for four is not `IIII`. Instead, the number four is written as `IV`. Because the one is before the five we subtract it making four. The same principle applies to the number nine, which is written as `IX`. There are six instances where subtraction is used:

- `I` can be placed before `V` (5) and `X` (10) to make 4 and 9.
- `X` can be placed before `L` (50) and `C` (100) to make 40 and 90.
- `C` can be placed before `D` (500) and `M` (1000) to make 400 and 900.

Given a roman numeral, convert it to an integer.

**Example 1:**

    Input: s = "III"
    Output: 3
    Explanation: III = 3.

**Example 2:**

    Input: s = "LVIII"
    Output: 58
    Explanation: L = 50, V = 5, III = 3.

**Example 3:**

    Input: s = "MCMXCIV"
    Output: 1994
    Explanation: M = 1000, CM = 900, XC = 90 and IV = 4.

**Constraints:**

- `1 <= s.length <= 15`
- `s` contains only the characters `('I', 'V', 'X', 'L', 'C', 'D', 'M')`.
- It is **guaranteed** that `s` is a valid roman numeral in the range `[1, 3999]`.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Hash map lookup + single scan (subtract when a smaller value precedes a larger one)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (rewrite pairs, then sum) | O(n) | O(n) |
| 2️⃣ Optimized (one pass, look ahead) | O(n) | O(1) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"Normally symbols go largest to smallest and I just add them. The only twist is the six subtractive pairs, where a smaller symbol sits **before** a larger one."*

- Input always valid? → **Yes**, so no validation needed (no `IIII`, no `IC`).
- Uppercase only? → **Yes.**
- Range 1–3999 → length ≤ 15, so performance isn't the concern here. **Clean, correct logic is.**

### Step 1 · Brute force

**Idea:** Treat the six subtractive pairs as special tokens. Replace them with additive equivalents (`IV` → `IIII`, `CM` → `DCCCC`, …), then add up every symbol.

🗣️ *"A straightforward approach: hard-code the six subtractive pairs, rewrite them into purely additive form, then sum the symbols. It works, but it needs a table of special cases and builds new strings. I'd rather find the general rule behind those six cases."*

In [1]:
class SolutionBrute:
    VALUES = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
    EXPAND = {"IV": "IIII", "IX": "VIIII", "XL": "XXXX", "XC": "LXXXX", "CD": "CCCC", "CM": "DCCCC"}

    def romanToInt(self, s: str) -> int:
        for pair, additive in self.EXPAND.items():
            s = s.replace(pair, additive)
        return sum(self.VALUES[ch] for ch in s)

### Step 2 · Optimize

**Bottleneck:** It's not speed (both are O(n)). It's the **six hard-coded special cases** and the extra string copies.

**Key insight: one general rule covers all six cases.**

> If a symbol's value is **less than the value of the symbol right after it**, subtract it. Otherwise add it.

`IV`: I(1) < V(5) → −1 + 5 = 4. `CM`: C(100) < M(1000) → −100 + 1000 = 900. In a valid numeral, the subtractive pairs are exactly the places where a value is smaller than its right neighbour.

🗣️ *"Instead of listing the six pairs, I'll use the rule that produces them: if the current symbol is smaller than the next one, subtract it, otherwise add it. I use a dict for the symbol values and make one pass, O(n) time and O(1) space."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| Hard-code pairs + string replace | ⚠️ | Works, but it's six special cases, builds new strings, and is easy to typo. |
| `if/elif` chain for the 7 symbols | ⚠️ | Verbose. A lookup table is cleaner and less error-prone. |
| **Dict `symbol → value` + compare with next** | ✅ | O(1) lookup, one general rule, a single pass. |
| Scan **right-to-left**, subtract if smaller than the previous value seen | ✅ (equivalent) | Same complexity. Some find it more natural ("a smaller value to the left of a larger one gets subtracted"). |
| Regex tokenising | ❌ | Overkill for a 15-char string with one simple rule. |

In [2]:
class Solution:
    VALUES = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}

    def romanToInt(self, s: str) -> int:
        total = 0
        for i, ch in enumerate(s):
            v = self.VALUES[ch]
            if i + 1 < len(s) and v < self.VALUES[s[i + 1]]:
                total -= v          # subtractive position (I before V/X, etc.)
            else:
                total += v
        return total

### Step 3 · Toy example walkthrough

`s = "MCMXCIV"`

| i | ch | value | next value | value < next? | action | total |
|---|---|---|---|---|---|---|
| 0 | M | 1000 | C = 100 | ❌ | +1000 | 1000 |
| 1 | C | 100 | M = 1000 | ✅ | **−100** | 900 |
| 2 | M | 1000 | X = 10 | ❌ | +1000 | 1900 |
| 3 | X | 10 | C = 100 | ✅ | **−10** | 1890 |
| 4 | C | 100 | I = 1 | ❌ | +100 | 1990 |
| 5 | I | 1 | V = 5 | ✅ | **−1** | 1989 |
| 6 | V | 5 | (none) | ❌ | +5 | **1994** ✅ |

The three subtractions match the three subtractive pairs CM, XC and IV, without listing any pairs explicitly.

In [3]:
cases = [
    ("III", 3),
    ("LVIII", 58),
    ("MCMXCIV", 1994),
    ("IV", 4),
    ("IX", 9),
    ("XL", 40),
    ("CD", 400),
    ("MMMCMXCIX", 3999),
    ("I", 1),
]
for s, expected in cases:
    assert SolutionBrute().romanToInt(s) == expected
    assert Solution().romanToInt(s) == expected
    print(s, "->", expected)

# round-trip check over the full range using a reference encoder
def to_roman(num):
    table = [(1000, "M"), (900, "CM"), (500, "D"), (400, "CD"), (100, "C"), (90, "XC"),
             (50, "L"), (40, "XL"), (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")]
    out = []
    for v, sym in table:
        q, num = divmod(num, v)
        out.append(sym * q)
    return "".join(out)

for n in range(1, 4000):
    assert Solution().romanToInt(to_roman(n)) == n
print("All tests passed ✅ (including all of 1..3999)")

III -> 3
LVIII -> 58
MCMXCIV -> 1994
IV -> 4
IX -> 9
XL -> 40
CD -> 400
MMMCMXCIX -> 3999
I -> 1
All tests passed ✅ (including all of 1..3999)


### Step 4 · Wrap up

| | Time | Space | Why |
|---|---|---|---|
| Rewrite pairs + sum | O(n) | O(n) | Six `replace` passes plus new strings |
| **Compare with next** | **O(n)** | **O(1)** | One pass and a fixed 7-entry dict |

**Complexity, explained:** one loop over the string, with an O(1) dict lookup and comparison per character → O(n). The dict always has 7 entries regardless of input → O(1) space. (Since n ≤ 15, everything here is effectively constant, but say the general complexity anyway.)

**Edge cases:** single symbol, the max value `MMMCMXCIX` (3999), every subtractive pair.

**Follow-ups:**
- *Validate the input?* → You'd need extra rules (at most 3 repeats, only I/X/C may subtract, only from the next two higher symbols). The compare-with-next rule alone accepts invalid strings like `IC`.
- *Reverse direction* → Integer to Roman (next problem): greedy with the 13-entry value table.

🗣️ **30-second recap:** *"Map each symbol to its value with a dict. Walk the string once: if a symbol is smaller than the one after it, subtract it, otherwise add it. That single rule covers all six subtractive cases without hard-coding them. O(n) time, O(1) space."*